# Phase 03 — Normalization and Feature Design

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Design stable normalization and feature rules for skills, experience, language, roles, and text before generating pairs.

This notebook is the Phase 3 source of truth. It writes `reports/phase_03_normalization_feature_design.json` so pair generation, baseline evaluation, and later model experiments share the same normalization contract.

## Purpose
Document and verify Phase 03 — Normalization and Feature Design in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 03.normalization.feature.design notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 03 — Normalization and Feature Design.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Contract boundary

Phase 3 defines deterministic feature rules only. It does not generate training pairs, fit models, calibrate scores, or change backend/API behavior.

Model/training may use normalized experience, normalized skills, language slices, role text, requirement text, and source-only profile/CV/job text. Backend/API wrapper remains owner of auth, persistence, final copy, hydrated job details, user preference filtering, and request orchestration.


## Shared setup

### Purpose
Define deterministic paths, load audited data, and create validation helpers used by every Phase 3 step.

### Required input
Repository root with `TODOS.md`, `reports/phase_01_data_audit_contracts.json`, `reports/phase_02_label_schema_baselines.json`, `legacy/dataset/indotech_job_cleaned.csv`, and `legacy/dataset/techtalent_profile_cleaned.csv`.

### Action
Load audited source snapshots, collect observed values, and define helpers for writing the Phase 3 report. No pair generation, label creation, model training, or artifact overwrite happens in this notebook.

### Expected output
Shared constants and helpers for normalization policies, observed-value coverage checks, feature-quality metrics, and final report writing.

### Verification
Setup must run with standard Python plus pandas. Generated output must be limited to `reports/phase_03_normalization_feature_design.json`.


In [7]:
from __future__ import annotations

import json
import re
import string
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd().resolve())
REPORTS = ROOT / "reports"
PHASE1_REPORT = REPORTS / "phase_01_data_audit_contracts.json"
PHASE2_REPORT = REPORTS / "phase_02_label_schema_baselines.json"
PHASE3_REPORT = REPORTS / "phase_03_normalization_feature_design.json"
JOBS_CSV = ROOT / "legacy" / "dataset" / "indotech_job_cleaned.csv"
PROFILES_CSV = ROOT / "legacy" / "dataset" / "techtalent_profile_cleaned.csv"


def load_json(path: Path) -> dict[str, Any]:
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def value_counts(series: pd.Series) -> dict[str, int]:
    return {str(k): int(v) for k, v in series.fillna("<NA>").value_counts(dropna=False).items()}


def non_empty_rate(series: pd.Series) -> float:
    cleaned = series.fillna("").astype(str).str.strip()
    return float((cleaned != "").mean()) if len(cleaned) else 0.0


phase1_report = load_json(PHASE1_REPORT)
phase2_report = load_json(PHASE2_REPORT)
jobs = pd.read_csv(JOBS_CSV)
profiles = pd.read_csv(PROFILES_CSV)

observed_profile_experience = sorted(str(v) for v in profiles["Experience"].dropna().unique())
observed_job_experience = sorted(str(v) for v in jobs["experience_level"].dropna().unique())
observed_language_signals = sorted(str(v) for v in jobs["language_signal"].dropna().unique())

print({
    "jobs_rows": int(len(jobs)),
    "profiles_rows": int(len(profiles)),
    "profile_experience_values": observed_profile_experience,
    "job_experience_values": observed_job_experience,
    "language_signals": observed_language_signals,
})


{'jobs_rows': 2073, 'profiles_rows': 69929, 'profile_experience_values': ['1-2 years', '3-5 years', '5+ years', 'Fresher'], 'job_experience_values': ['ENTRY_LEVEL', 'JUNIOR', 'LEAD', 'MID_LEVEL', 'SENIOR'], 'language_signals': ['EN', 'ID', 'MIXED', 'UNKNOWN']}


## Step 3.1 — Experience normalization

### Purpose
Define explicit mappings for Fresher, 1-2 years, 3-5 years, 5+ years, ENTRY_LEVEL, JUNIOR, MID_LEVEL, SENIOR, LEAD, and MANAGER.

### Required input
Use audited profile `Experience` values, job `experience_level` values, Phase 2 experience-match label policy, and current source distributions. Observed source values must be checked against the mapping before pair generation.

### Action
Create a versioned experience normalization table with minimum years, maximum years, representative midpoint, canonical band, seniority order, unknown policy, and allowed source aliases. Treat unmapped values as `UNKNOWN_EXPERIENCE` with an explicit quality flag instead of defaulting to neutral values silently.

### Expected output
A coverage-checked mapping table and report fields showing every observed profile/job experience value maps deterministically.

### Verification
All observed values in the current source snapshot must be covered. The required roadmap values must also exist in the mapping, including `MANAGER` even if it is not present in the current snapshot.


In [8]:
experience_normalization = {
    "Fresher": {
        "canonical_band": "entry",
        "min_years": 0.0,
        "max_years": 0.0,
        "representative_years": 0.0,
        "seniority_order": 0,
        "source": "profile.Experience",
        "notes": "No professional experience evidence; use as entry-level candidate signal.",
    },
    "1-2 years": {
        "canonical_band": "junior",
        "min_years": 1.0,
        "max_years": 2.0,
        "representative_years": 1.5,
        "seniority_order": 1,
        "source": "profile.Experience",
        "notes": "Early professional experience bucket.",
    },
    "3-5 years": {
        "canonical_band": "mid_level",
        "min_years": 3.0,
        "max_years": 5.0,
        "representative_years": 4.0,
        "seniority_order": 2,
        "source": "profile.Experience",
        "notes": "Mid-level candidate evidence; may match MID_LEVEL and lower SENIOR requirements depending on role context.",
    },
    "5+ years": {
        "canonical_band": "senior_or_above",
        "min_years": 5.0,
        "max_years": None,
        "representative_years": 5.0,
        "seniority_order": 3,
        "source": "profile.Experience",
        "notes": "Open-ended senior experience bucket; do not infer lead/manager without role evidence.",
    },
    "ENTRY_LEVEL": {
        "canonical_band": "entry",
        "min_years": 0.0,
        "max_years": 1.0,
        "representative_years": 0.5,
        "seniority_order": 0,
        "source": "job.experience_level",
        "notes": "Entry-level job requirement.",
    },
    "JUNIOR": {
        "canonical_band": "junior",
        "min_years": 1.0,
        "max_years": 2.0,
        "representative_years": 1.5,
        "seniority_order": 1,
        "source": "job.experience_level",
        "notes": "Junior job requirement.",
    },
    "MID_LEVEL": {
        "canonical_band": "mid_level",
        "min_years": 3.0,
        "max_years": 5.0,
        "representative_years": 4.0,
        "seniority_order": 2,
        "source": "job.experience_level",
        "notes": "Mid-level job requirement.",
    },
    "SENIOR": {
        "canonical_band": "senior",
        "min_years": 5.0,
        "max_years": 8.0,
        "representative_years": 6.5,
        "seniority_order": 3,
        "source": "job.experience_level",
        "notes": "Senior individual-contributor requirement.",
    },
    "LEAD": {
        "canonical_band": "lead",
        "min_years": 6.0,
        "max_years": 10.0,
        "representative_years": 8.0,
        "seniority_order": 4,
        "source": "job.experience_level",
        "notes": "Lead role requirement; should also require role/title evidence before strong match.",
    },
    "MANAGER": {
        "canonical_band": "manager",
        "min_years": 5.0,
        "max_years": 10.0,
        "representative_years": 7.5,
        "seniority_order": 4,
        "source": "job.experience_level",
        "notes": "Management requirement; do not infer from years alone without management role evidence.",
    },
}

experience_unknown_policy = {
    "canonical_value": "UNKNOWN_EXPERIENCE",
    "feature_flag": "unknown_experience",
    "score_policy": "Do not silently default to neutral. Record unknown_experience=true and exclude from strict experience-match labels until reviewed.",
    "pair_generation_action": "Block pair-generation release if any observed source value is unmapped.",
}

required_experience_values = {
    "Fresher", "1-2 years", "3-5 years", "5+ years", "ENTRY_LEVEL", "JUNIOR", "MID_LEVEL", "SENIOR", "LEAD", "MANAGER"
}
observed_experience_values = set(observed_profile_experience) | set(observed_job_experience)
missing_required_experience = sorted(required_experience_values - set(experience_normalization))
unmapped_observed_experience = sorted(observed_experience_values - set(experience_normalization))

experience_coverage = {
    "observed_profile_values": observed_profile_experience,
    "observed_job_values": observed_job_experience,
    "profile_counts": value_counts(profiles["Experience"]),
    "job_counts": value_counts(jobs["experience_level"]),
    "missing_required_values": missing_required_experience,
    "unmapped_observed_values": unmapped_observed_experience,
    "all_observed_values_mapped": not unmapped_observed_experience,
}

assert not missing_required_experience, missing_required_experience
assert not unmapped_observed_experience, unmapped_observed_experience
print(json.dumps(experience_coverage, indent=2))


{
  "observed_profile_values": [
    "1-2 years",
    "3-5 years",
    "5+ years",
    "Fresher"
  ],
  "observed_job_values": [
    "ENTRY_LEVEL",
    "JUNIOR",
    "LEAD",
    "MID_LEVEL",
    "SENIOR"
  ],
  "profile_counts": {
    "5+ years": 17569,
    "Fresher": 17503,
    "3-5 years": 17445,
    "1-2 years": 17412
  },
  "job_counts": {
    "ENTRY_LEVEL": 1097,
    "JUNIOR": 391,
    "MID_LEVEL": 246,
    "LEAD": 204,
    "SENIOR": 135
  },
  "missing_required_values": [],
  "unmapped_observed_values": [],
  "all_observed_values_mapped": true
}


## Step 3.2 — Skill normalization

### Purpose
Document lowercase rules, punctuation trimming, alias handling, Indonesian/English variants, framework variants, and unknown-skill handling.

### Required input
Use audited job skill fields (`skills_clean`, `skills_top_10_names`), profile skill fields (`Skills`, `Required_Skills`), Phase 2 skill-overlap label policy, and known framework spelling variants.

### Action
Define deterministic skill parsing and canonicalization rules. Lowercase first, trim whitespace and list punctuation, normalize separators, preserve meaningful technical punctuation inside tokens (`c++`, `c#`, `.net`, `node.js`, `next.js`, `ci/cd`), then apply a versioned alias table. Indonesian and English variants map to the same canonical skill only when meaning is unambiguous. Unknown skills remain as normalized tokens with `unknown_skill=true`; they are not dropped.

### Expected output
A skill-normalization policy, alias dictionary, parser rules, current snapshot quality metrics, and examples of top normalized skills.

### Verification
Alias policy must be explicit and reviewable. Empty skill lists and unknown aliases must be measurable before pair generation.


In [9]:
SKILL_SEPARATOR_RE = re.compile(r"\s*(?:\||,|;|\n|\t)\s*")
TRIM_CHARS = string.whitespace + "'\"`“”‘’()[]{}<>:"

skill_aliases = {
    "js": "javascript",
    "javascript": "javascript",
    "java script": "javascript",
    "ts": "typescript",
    "typescript": "typescript",
    "type script": "typescript",
    "react": "react",
    "react.js": "react",
    "react js": "react",
    "reactjs": "react",
    "next.js": "next.js",
    "next js": "next.js",
    "nextjs": "next.js",
    "node.js": "node.js",
    "node js": "node.js",
    "nodejs": "node.js",
    "vue.js": "vue.js",
    "vue js": "vue.js",
    "vuejs": "vue.js",
    "ci/cd": "ci/cd",
    "ci cd": "ci/cd",
    "cicd": "ci/cd",
    "gitlab ci": "gitlab ci/cd",
    "gitlab ci/cd": "gitlab ci/cd",
    "powerbi": "power bi",
    "power bi": "power bi",
    "google bigquery": "google bigquery",
    "bigquery": "google bigquery",
    "postgres": "postgresql",
    "postgresql": "postgresql",
    "postgre sql": "postgresql",
    "mysql": "mysql",
    "sql": "sql",
    "machine learning": "machine learning",
    "ml": "machine learning",
    "pembelajaran mesin": "machine learning",
    "deep learning": "deep learning",
    "artificial intelligence": "artificial intelligence",
    "ai": "artificial intelligence",
    "kecerdasan buatan": "artificial intelligence",
    "data analysis": "data analysis",
    "analisis data": "data analysis",
    "data analytics": "data analytics",
    "visualization": "data visualization",
    "visualisation": "data visualization",
    "visualisasi data": "data visualization",
    "api": "api",
    "apis": "api",
    "rest api": "rest api",
    "docker": "docker",
    "kubernetes": "kubernetes",
    "k8s": "kubernetes",
    "tailwind": "tailwind css",
    "tailwind css": "tailwind css",
    "html": "html",
    "css": "css",
    "python": "python",
    "pytorch": "pytorch",
    "tensorflow": "tensorflow",
    "git": "git",
    "gitlab": "gitlab",
}

skill_normalization_policy = {
    "schema_version": "skill-normalization-v1",
    "case_rule": "Convert Unicode text to lowercase before alias lookup.",
    "separator_rule": "Split list fields on pipe, comma, semicolon, tab, or newline. Keep slash/dot/plus/hash inside tokens when meaningful.",
    "trim_rule": "Trim surrounding whitespace, quotes, brackets, braces, parentheses, angle brackets, and colon characters.",
    "punctuation_rule": "Collapse repeated whitespace; normalize common spacing around slash, dot, plus, and hash; do not remove technical punctuation from c++, c#, .net, node.js, next.js, or ci/cd.",
    "alias_rule": "Apply exact alias dictionary after lowercase and punctuation cleanup. Alias changes require version bump and review.",
    "id_en_rule": "Map Indonesian and English variants only when the technical meaning is unambiguous; otherwise preserve normalized source token and flag for review.",
    "framework_variant_rule": "Framework spellings such as react.js/reactjs and next.js/nextjs resolve to one canonical token.",
    "unknown_skill_policy": "Unknown non-empty tokens are preserved as canonical_skill=normalized_token with unknown_skill=true; never drop them silently.",
}


def normalize_skill_token(token: Any) -> dict[str, Any]:
    raw = "" if token is None else str(token)
    lowered = raw.strip().lower()
    trimmed = lowered.strip(TRIM_CHARS)
    trimmed = re.sub(r"\s+", " ", trimmed)
    trimmed = re.sub(r"\s*/\s*", "/", trimmed)
    trimmed = re.sub(r"\s*\.\s*", ".", trimmed)
    trimmed = re.sub(r"\s*\+\s*", "+", trimmed)
    trimmed = re.sub(r"\s*#\s*", "#", trimmed)
    canonical = skill_aliases.get(trimmed, trimmed)
    return {
        "raw": raw,
        "normalized": trimmed,
        "canonical": canonical,
        "is_empty": trimmed == "",
        "alias_applied": bool(trimmed and canonical != trimmed),
        "unknown_skill": bool(trimmed and trimmed not in skill_aliases),
    }


def parse_skill_list(value: Any) -> list[dict[str, Any]]:
    if value is None or pd.isna(value):
        return []
    parts = [p for p in SKILL_SEPARATOR_RE.split(str(value)) if p.strip()]
    return [row for row in (normalize_skill_token(part) for part in parts) if not row["is_empty"]]

skill_sources = {
    "jobs.skills_clean": jobs["skills_clean"],
    "jobs.skills_top_10_names": jobs["skills_top_10_names"],
    "profiles.Skills": profiles["Skills"],
    "profiles.Required_Skills": profiles["Required_Skills"],
}

skill_quality_by_source = {}
canonical_counter: Counter[str] = Counter()
alias_counter: Counter[str] = Counter()
unknown_counter: Counter[str] = Counter()
for name, series in skill_sources.items():
    parsed_rows = [parse_skill_list(value) for value in series]
    list_lengths = [len(row) for row in parsed_rows]
    flat = [item for row in parsed_rows for item in row]
    canonical_counter.update(item["canonical"] for item in flat)
    alias_counter.update(item["normalized"] for item in flat if item["alias_applied"])
    unknown_counter.update(item["canonical"] for item in flat if item["unknown_skill"])
    skill_quality_by_source[name] = {
        "rows": int(len(series)),
        "empty_skill_rows": int(sum(length == 0 for length in list_lengths)),
        "empty_skill_rate": float(sum(length == 0 for length in list_lengths) / len(list_lengths)) if list_lengths else 0.0,
        "total_skill_tokens": int(sum(list_lengths)),
        "unique_canonical_skills": int(len({item["canonical"] for item in flat})),
        "alias_applied_tokens": int(sum(1 for item in flat if item["alias_applied"])),
        "unknown_skill_tokens": int(sum(1 for item in flat if item["unknown_skill"])),
    }

skill_quality_summary = {
    "by_source": skill_quality_by_source,
    "alias_dictionary_size": int(len(skill_aliases)),
    "top_canonical_skills": dict(canonical_counter.most_common(25)),
    "top_alias_inputs": dict(alias_counter.most_common(25)),
    "top_unknown_skill_tokens_for_review": dict(unknown_counter.most_common(25)),
}

assert skill_aliases["react.js"] == "react"
assert skill_aliases["reactjs"] == "react"
assert skill_aliases["nextjs"] == "next.js"
assert normalize_skill_token(" React.Js ")["canonical"] == "react"
assert normalize_skill_token("CI / CD")["canonical"] == "ci/cd"
print(json.dumps(skill_quality_summary, indent=2)[:4000])


{
  "by_source": {
    "jobs.skills_clean": {
      "rows": 2073,
      "empty_skill_rows": 0,
      "empty_skill_rate": 0.0,
      "total_skill_tokens": 7405,
      "unique_canonical_skills": 1334,
      "alias_applied_tokens": 116,
      "unknown_skill_tokens": 5939
    },
    "jobs.skills_top_10_names": {
      "rows": 2073,
      "empty_skill_rows": 0,
      "empty_skill_rate": 0.0,
      "total_skill_tokens": 7405,
      "unique_canonical_skills": 1334,
      "alias_applied_tokens": 116,
      "unknown_skill_tokens": 5939
    },
    "profiles.Skills": {
      "rows": 69929,
      "empty_skill_rows": 0,
      "empty_skill_rate": 0.0,
      "total_skill_tokens": 535644,
      "unique_canonical_skills": 44,
      "alias_applied_tokens": 26315,
      "unknown_skill_tokens": 265398
    },
    "profiles.Required_Skills": {
      "rows": 69929,
      "empty_skill_rows": 0,
      "empty_skill_rate": 0.0,
      "total_skill_tokens": 349645,
      "unique_canonical_skills": 29,
      "alias

## Step 3.3 — Language normalization

### Purpose
Define ID, EN, MIXED, and UNKNOWN assignment rules and the minimum evidence needed for each category.

### Required input
Use audited `language_signal`, job title/description/requirements text, and future CV/profile text. Current language counts are source-quality evidence, not production calibration.

### Action
Define assignment rules with minimum evidence. Prefer trusted source language metadata when it is one of `ID`, `EN`, `MIXED`, or `UNKNOWN`; otherwise derive language from visible text markers. Use `UNKNOWN` when text is too short, marker evidence is weak, or detected signals conflict below threshold.

### Expected output
A language-normalization policy and current snapshot counts for language-quality reporting.

### Verification
Every source language value must be in the allowed set. Unknown and mixed rates must be reported before pair generation and slice evaluation.


In [10]:
language_normalization_policy = {
    "allowed_values": ["ID", "EN", "MIXED", "UNKNOWN"],
    "trusted_source_rule": "Accept source language_signal only when it is exactly ID, EN, MIXED, or UNKNOWN.",
    "minimum_text_rule": "If combined evidence text has fewer than 40 non-whitespace characters, assign UNKNOWN unless trusted source is ID/EN/MIXED.",
    "id_rule": "Assign ID when Indonesian marker count is at least 2 and English marker count is below 2, or trusted source says ID.",
    "en_rule": "Assign EN when English marker count is at least 2 and Indonesian marker count is below 2, or trusted source says EN.",
    "mixed_rule": "Assign MIXED when both Indonesian and English marker counts are at least 2, or trusted source says MIXED.",
    "unknown_rule": "Assign UNKNOWN when evidence is missing, marker counts are below threshold, or source value is not in the allowed set.",
    "slice_policy": "Report metrics separately for ID, EN, MIXED, and UNKNOWN. Do not hide UNKNOWN inside EN.",
}

id_markers = {
    "dan", "dengan", "untuk", "pengalaman", "keterampilan", "kemampuan", "lulusan", "kerja", "mengelola", "membangun", "analisis"
}
en_markers = {
    "and", "with", "for", "experience", "skills", "requirements", "developer", "engineer", "manage", "build", "analysis"
}


def language_from_text(text: Any, source_signal: Any = None) -> dict[str, Any]:
    source = "" if source_signal is None or pd.isna(source_signal) else str(source_signal).strip().upper()
    if source in {"ID", "EN", "MIXED"}:
        return {"language": source, "source": "trusted_source", "id_marker_count": None, "en_marker_count": None}
    raw = "" if text is None or pd.isna(text) else str(text)
    tokens = re.findall(r"[a-zA-Z]+", raw.lower())
    id_count = sum(1 for token in tokens if token in id_markers)
    en_count = sum(1 for token in tokens if token in en_markers)
    if len("".join(tokens)) < 40:
        lang = "UNKNOWN"
    elif id_count >= 2 and en_count >= 2:
        lang = "MIXED"
    elif id_count >= 2:
        lang = "ID"
    elif en_count >= 2:
        lang = "EN"
    else:
        lang = "UNKNOWN"
    return {"language": lang, "source": "text_markers", "id_marker_count": id_count, "en_marker_count": en_count}

allowed_language_values = set(language_normalization_policy["allowed_values"])
unexpected_language_values = sorted(set(observed_language_signals) - allowed_language_values)
language_counts = value_counts(jobs["language_signal"])
job_language_quality = {
    "observed_values": observed_language_signals,
    "unexpected_values": unexpected_language_values,
    "counts": language_counts,
    "unknown_rate": float((jobs["language_signal"].fillna("UNKNOWN") == "UNKNOWN").mean()),
    "mixed_rate": float((jobs["language_signal"].fillna("UNKNOWN") == "MIXED").mean()),
}

assert not unexpected_language_values, unexpected_language_values
print(json.dumps(job_language_quality, indent=2))


{
  "observed_values": [
    "EN",
    "ID",
    "MIXED",
    "UNKNOWN"
  ],
  "unexpected_values": [],
  "counts": {
    "EN": 1426,
    "UNKNOWN": 602,
    "MIXED": 23,
    "ID": 22
  },
  "unknown_rate": 0.2904003859141341,
  "mixed_rate": 0.011095031355523395
}


## Step 3.4 — Text feature construction

### Purpose
Describe how profile text, CV text, job text, role text, requirements, and skills are combined before embedding.

### Required input
Use source-only profile fields, parsed CV text when available, source job fields, normalized skills, normalized roles, and Phase 2 leakage controls. Do not use generated labels, wrapper summaries, application outcomes, or hydrated job detail copy as embedding text.

### Action
Define deterministic text sections, order, separators, de-duplication, length caps, missing-field behavior, and feature dictionary. Construct text from evidence fields only and keep structured scalar features separate from embedding text.

### Expected output
A feature-construction contract for profile/CV text, job text, role text, requirement text, normalized skill lists, scalar quality flags, and later embedding inputs.

### Verification
Constructed text must exclude identifiers, `fit_score`, product copy, `topActionables`, `sectionReviews`, and backend-hydrated details. Empty text rates must be measurable.


In [11]:
text_feature_construction = {
    "schema_version": "text-feature-construction-v1",
    "separator": "\n---\n",
    "dedupe_rule": "Normalize whitespace and remove duplicate non-empty sections while preserving deterministic section order.",
    "length_policy": "Keep source text fields intact for audit; later embedding jobs may apply model-specific token truncation after logging original character counts.",
    "excluded_fields": [
        "profile_id", "job_id", "ID", "jobFitAlignment.summary", "topActionables", "sectionReviews",
        "overallImpression product copy", "fit_score", "application outcome", "bookmark status", "hydrated job details",
    ],
    "profile_text_sections": [
        {"section": "role", "fields": ["Job_Role"], "required": False},
        {"section": "skills", "fields": ["Skills"], "required": True},
        {"section": "projects", "fields": ["Projects"], "required": False},
        {"section": "experience", "fields": ["Experience"], "required": True},
    ],
    "cv_text_sections": [
        {"section": "parsed_cv_text", "fields": ["parsed_text"], "required": True},
        {"section": "detected_sections", "fields": ["section_headings"], "required": False},
        {"section": "cv_skills", "fields": ["extracted_skills"], "required": False},
    ],
    "job_text_sections": [
        {"section": "title", "fields": ["title", "normalized_title"], "required": True},
        {"section": "category", "fields": ["category"], "required": False},
        {"section": "requirements", "fields": ["requirement_summary", "requirements_concat"], "required": True},
        {"section": "skills", "fields": ["skills_clean", "skills_top_10_names"], "required": True},
        {"section": "description", "fields": ["description"], "required": True},
        {"section": "experience_level", "fields": ["experience_level"], "required": True},
    ],
    "scalar_features_kept_outside_embedding_text": [
        "skill_overlap", "experience_gap_years", "seniority_gap", "role_family_match", "requirement_coverage",
        "language", "unknown_language", "unknown_experience", "empty_skills", "empty_text",
    ],
}


def clean_text(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def join_sections(row: pd.Series, section_specs: list[dict[str, Any]]) -> str:
    parts: list[str] = []
    seen: set[str] = set()
    for spec in section_specs:
        values = [clean_text(row.get(field, "")) for field in spec["fields"]]
        body = " | ".join(value for value in values if value)
        if not body:
            continue
        section_text = f"{spec['section']}: {body}"
        dedupe_key = section_text.lower()
        if dedupe_key not in seen:
            seen.add(dedupe_key)
            parts.append(section_text)
    return text_feature_construction["separator"].join(parts)

profile_text_sample = profiles.head(1000).apply(
    lambda row: join_sections(row, text_feature_construction["profile_text_sections"]), axis=1
)
job_text_sample = jobs.apply(
    lambda row: join_sections(row, text_feature_construction["job_text_sections"]), axis=1
)

text_feature_quality = {
    "profile_text_sample_rows": int(len(profile_text_sample)),
    "profile_text_empty_rate_sample": float((profile_text_sample.str.strip() == "").mean()),
    "profile_text_min_chars_sample": int(profile_text_sample.str.len().min()),
    "profile_text_median_chars_sample": float(profile_text_sample.str.len().median()),
    "job_text_rows": int(len(job_text_sample)),
    "job_text_empty_rate": float((job_text_sample.str.strip() == "").mean()),
    "job_text_min_chars": int(job_text_sample.str.len().min()),
    "job_text_median_chars": float(job_text_sample.str.len().median()),
    "excluded_fields_verified": text_feature_construction["excluded_fields"],
}

assert "fit_score" in text_feature_construction["excluded_fields"]
assert text_feature_quality["job_text_empty_rate"] == 0.0
print(json.dumps(text_feature_quality, indent=2))


{
  "profile_text_sample_rows": 1000,
  "profile_text_empty_rate_sample": 0.0,
  "profile_text_min_chars_sample": 145,
  "profile_text_median_chars_sample": 189.0,
  "job_text_rows": 2073,
  "job_text_empty_rate": 0.0,
  "job_text_min_chars": 347,
  "job_text_median_chars": 659.0,
  "excluded_fields_verified": [
    "profile_id",
    "job_id",
    "ID",
    "jobFitAlignment.summary",
    "topActionables",
    "sectionReviews",
    "overallImpression product copy",
    "fit_score",
    "application outcome",
    "bookmark status",
    "hydrated job details"
  ]
}


## Step 3.5 — Missing and unknown report

### Purpose
Define metrics for unknown language rate, unknown experience rate, empty skills rate, and empty text rate.

### Required input
Use Phase 3 normalization policies, current audited source data, and feature-construction outputs. Pair generation must publish these metrics for each future dataset version.

### Action
Define durable metric names, formulas, severity thresholds, ownership, and required slices. Generate current snapshot metrics for jobs and profiles where source fields exist.

### Expected output
A feature-quality report contract plus current snapshot quality metrics that later phases can compare against.

### Verification
The report must include unknown language rate, unknown experience rate, empty skills rate, and empty text rate. Acceptance checks must fail if observed experience values are unmapped or feature-quality metric definitions are missing.


In [12]:
feature_quality_metric_definitions = [
    {
        "metric": "unknown_language_rate",
        "formula": "rows with normalized_language == UNKNOWN / total rows",
        "owner": "normalization",
        "required_slices": ["dataset", "role_family", "experience_band"],
        "warn_threshold": 0.10,
        "block_threshold": 0.30,
    },
    {
        "metric": "unknown_experience_rate",
        "formula": "rows with unknown_experience == true / total rows",
        "owner": "normalization",
        "required_slices": ["dataset", "source_field", "role_family"],
        "warn_threshold": 0.01,
        "block_threshold": 0.05,
    },
    {
        "metric": "empty_skills_rate",
        "formula": "rows with zero parsed skills / total rows",
        "owner": "feature_builder",
        "required_slices": ["dataset", "skill_source", "role_family", "language"],
        "warn_threshold": 0.05,
        "block_threshold": 0.15,
    },
    {
        "metric": "empty_text_rate",
        "formula": "rows with constructed text length == 0 / total rows",
        "owner": "feature_builder",
        "required_slices": ["dataset", "text_source", "language"],
        "warn_threshold": 0.01,
        "block_threshold": 0.05,
    },
    {
        "metric": "unknown_skill_token_rate",
        "formula": "parsed skill tokens without alias dictionary match / total parsed skill tokens",
        "owner": "normalization",
        "required_slices": ["dataset", "skill_source"],
        "warn_threshold": 0.50,
        "block_threshold": 0.80,
    },
]

current_feature_quality_snapshot = {
    "jobs": {
        "rows": int(len(jobs)),
        "unknown_language_rate": float((jobs["language_signal"].fillna("UNKNOWN") == "UNKNOWN").mean()),
        "unknown_experience_rate": float((~jobs["experience_level"].fillna("").isin(experience_normalization)).mean()),
        "empty_skills_rate": float((jobs["skills_clean"].fillna("").astype(str).str.strip() == "").mean()),
        "empty_text_rate": text_feature_quality["job_text_empty_rate"],
    },
    "profiles": {
        "rows": int(len(profiles)),
        "unknown_language_rate": None,
        "unknown_language_note": "No audited profile language field exists in the current snapshot; infer later from profile/CV text and report separately.",
        "unknown_experience_rate": float((~profiles["Experience"].fillna("").isin(experience_normalization)).mean()),
        "empty_skills_rate": float((profiles["Skills"].fillna("").astype(str).str.strip() == "").mean()),
        "empty_text_rate_sample": text_feature_quality["profile_text_empty_rate_sample"],
    },
}

feature_quality_report_plan = {
    "schema_version": "feature-quality-report-v1",
    "required_metrics": feature_quality_metric_definitions,
    "required_outputs": [
        "overall metrics by dataset",
        "metrics by role family",
        "metrics by normalized language",
        "metrics by normalized experience band",
        "top unmapped experience values",
        "top unknown skill tokens for alias review",
        "empty text examples with source IDs stored separately from model features",
    ],
    "release_gate": "Pair generation must stop when unmapped observed experience values exist. Other metrics must be reported with warn/block thresholds before training.",
}

phase3_acceptance = {
    "no_known_experience_value_falls_through_silently": experience_coverage["all_observed_values_mapped"],
    "skill_alias_policy_is_documented": bool(skill_aliases and skill_normalization_policy["alias_rule"]),
    "feature_quality_metrics_are_defined": {row["metric"] for row in feature_quality_metric_definitions} >= {
        "unknown_language_rate", "unknown_experience_rate", "empty_skills_rate", "empty_text_rate"
    },
}
assert all(phase3_acceptance.values()), phase3_acceptance

phase3_report = {
    "schema_version": "phase-03-normalization-feature-design-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_snapshot": "legacy",
    "inputs": {
        "phase1_report": str(PHASE1_REPORT.relative_to(ROOT)),
        "phase2_report": str(PHASE2_REPORT.relative_to(ROOT)),
        "jobs_csv": str(JOBS_CSV.relative_to(ROOT)),
        "profiles_csv": str(PROFILES_CSV.relative_to(ROOT)),
        "todo_scope": "Phase 3 — Normalization and Feature Design",
    },
    "experience_normalization": experience_normalization,
    "experience_unknown_policy": experience_unknown_policy,
    "experience_coverage": experience_coverage,
    "skill_normalization_policy": skill_normalization_policy,
    "skill_aliases": skill_aliases,
    "skill_quality_summary": skill_quality_summary,
    "language_normalization_policy": language_normalization_policy,
    "language_quality": job_language_quality,
    "text_feature_construction": text_feature_construction,
    "text_feature_quality": text_feature_quality,
    "feature_quality_report_plan": feature_quality_report_plan,
    "current_feature_quality_snapshot": current_feature_quality_snapshot,
    "blocked_until_later_phases": [
        "Phase 4 must generate balanced pairs using these normalization policies and publish feature-quality metrics.",
        "Phase 5 must compare baselines using these normalized features before complex model training.",
        "Phase 7 must evaluate ATS text extraction quality before production ATS claims.",
        "Phase 10 must calibrate score meanings after feature and label distributions are stable.",
    ],
    "acceptance": phase3_acceptance,
}

write_json(PHASE3_REPORT, phase3_report)
print(f"Wrote {PHASE3_REPORT.relative_to(ROOT)}")
print(json.dumps(phase3_acceptance, indent=2))


Wrote reports/phase_03_normalization_feature_design.json
{
  "no_known_experience_value_falls_through_silently": true,
  "skill_alias_policy_is_documented": true,
  "feature_quality_metrics_are_defined": true
}


## Acceptance criteria

- [x] No known experience value falls through silently.
- [x] Skill alias policy is documented.
- [x] Feature-quality metrics are defined.


## Phase notes

Phase 3 defines normalization and feature-construction policies only. It intentionally does not generate pairs, create labels, train a model, tune thresholds, or claim production readiness.

Follow-up work:

- Phase 4 must use the experience and skill policies when creating balanced pairs and leakage-safe splits.
- Phase 5 must evaluate baselines using the normalized feature contract.
- Language `UNKNOWN` remains high in the job snapshot and must be reported as its own slice rather than folded into English.
- Skill aliases should be expanded only through reviewable versioned updates, using the unknown-token report as evidence.
